# Day 5 · 数据仓库屋顶：DWS + ADS 层

> **核心任务**：在 Day 4 建好的 DWD 明细层之上,继续建 DWS（汇总层）和 ADS（应用层）,产出可以直接喂前端看板的指标表

> **学习关键词**：主题域聚合、用户画像、Cohort 留存分析、漏斗分析、JSON 导出

## DWS 与 ADS 的本质区别

**这是数仓建模中最常被问的概念之一,搞清楚很重要**。

| 维度 | DWS（汇总层） | ADS（应用层） |
|---|---|---|
| **粒度** | 主题域 + 时间粒度（用户×日、内容×日） | 业务报表粒度（DAU 趋势、Top 100 榜） |
| **可复用性** | 高（多个 ADS 表会复用同一张 DWS 表） | 低（每张表对应一个具体看板） |
| **设计驱动** | 业务实体驱动 | 业务报表驱动 |
| **典型命名** | dws_user_daily, dws_movie_daily | ads_dau_trend, ads_top_movies |
| **数据量** | 中等（保留每个主题对象的明细） | 小（聚合后只剩最终数字） |

**类比**：DWS 是"半成品仓库"（切好的菜、煮好的肉）,ADS 是"出餐口"（按订单组装好的菜)。出新菜（新看板）时,只要从 DWS 拿现成的半成品,不用从原料重新做。

## 今天的产出

3 张 DWS + 5 张 ADS,**每一张都有具体的下游消费场景**（前端看板上的某个图）。

---
## 0. 启动 Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
import time

spark = SparkSession.builder \
    .appName("ContentDataPlatform-Day5") \
    .master("local[*]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# 项目路径(如果你的用户名不是 rein,改这一行)
PROJECT_ROOT = Path('/home/rein/projects/content-data-platform')
WAREHOUSE = PROJECT_ROOT / 'data' / 'warehouse'
DWD = WAREHOUSE / 'dwd'
DWS = WAREHOUSE / 'dws'
ADS = WAREHOUSE / 'ads'
ADS_JSON = PROJECT_ROOT / 'frontend' / 'data'  # 给前端用的 JSON 输出目录

for d in (DWS, ADS, ADS_JSON):
    d.mkdir(parents=True, exist_ok=True)

# 注册 DWD 层的表为视图
spark.read.parquet(str(DWD / 'fact_user_rating')).createOrReplaceTempView('dwd_fact_user_rating')
spark.read.parquet(str(DWD / 'dim_movie')).createOrReplaceTempView('dwd_dim_movie')

print(f'Spark version: {spark.version}')
print('✅ DWD 层已挂载,准备开始建上层')

26/06/23 06:09:49 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.31.129 instead (on interface ens33)
26/06/23 06:09:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 06:09:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/23 06:09:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.5.3
✅ DWD 层已挂载,准备开始建上层


---
# Part 1 · DWS 层建设

## 1.1 dws_user_daily —— 用户日活明细

**业务含义**：每个用户在每一天的所有行为聚合。

**下游消费者**：
- `ads_dau_monthly`（DAU 趋势图）会从这张表算 DISTINCT user
- `ads_user_funnel`（漏斗）会从这张表算用户活跃天数
- 未来可能的"用户画像 v2"也会复用这张表

**粒度**：(user_id, date) 一对一

**分区策略**：按 year 分区（跟事实表保持一致,后续查询可以走分区裁剪）

In [2]:
dws_user_daily = spark.sql("""
    SELECT 
        user_id,
        DATE(event_time) AS date,
        year,
        month,
        
        -- 行为聚合
        COUNT(*) AS rating_count,
        ROUND(AVG(rating), 2) AS avg_rating,
        MIN(rating) AS min_rating,
        MAX(rating) AS max_rating,
        
        -- 情绪分布(借用 DWD 层算好的 sentiment 字段)
        SUM(CASE WHEN sentiment = 'positive' THEN 1 ELSE 0 END) AS positive_count,
        SUM(CASE WHEN sentiment = 'neutral' THEN 1 ELSE 0 END) AS neutral_count,
        SUM(CASE WHEN sentiment = 'negative' THEN 1 ELSE 0 END) AS negative_count,
        
        -- 极端评分用户标识
        SUM(is_extreme) AS extreme_count,
        
        current_timestamp() AS dws_etl_time
    FROM dwd_fact_user_rating
    GROUP BY user_id, DATE(event_time), year, month
""")

# 写入,保持按 year 分区
start = time.time()
dws_user_daily.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet(str(DWS / 'dws_user_daily'))
print(f'✅ dws_user_daily 完成,耗时 {time.time()-start:.1f} 秒')

# 看一眼
spark.read.parquet(str(DWS / 'dws_user_daily')).createOrReplaceTempView('dws_user_daily')
spark.sql("SELECT * FROM dws_user_daily WHERE rating_count >= 10 LIMIT 5").show()

✅ dws_user_daily 完成,耗时 10.2 秒
+-------+----------+-----+------------+----------+----------+----------+--------------+-------------+--------------+-------------+--------------------+----+
|user_id|      date|month|rating_count|avg_rating|min_rating|max_rating|positive_count|neutral_count|negative_count|extreme_count|        dws_etl_time|year|
+-------+----------+-----+------------+----------+----------+----------+--------------+-------------+--------------+-------------+--------------------+----+
|  14023|2017-01-22|    1|          36|      4.53|       0.5|       5.0|            31|            3|             2|           30|2026-06-23 06:09:...|2017|
|     17|2017-01-27|    1|          29|      3.66|       2.5|       5.0|            15|            8|             6|            5|2026-06-23 06:09:...|2017|
|     93|2017-06-03|    6|         117|      3.92|       0.5|       5.0|            76|           28|            13|           32|2026-06-23 06:09:...|2017|
|  14094|2017-12-16|   12|  

## 1.2 dws_movie_daily —— 内容日热明细

**业务含义**：每部电影在每一天被打分的所有行为聚合。

**下游消费者**：
- `ads_top_movies`（Top 榜）会基于这张表做时间窗口聚合
- 未来"内容生命周期分析"会用这张表

In [3]:
dws_movie_daily = spark.sql("""
    SELECT 
        movie_id,
        DATE(event_time) AS date,
        year,
        month,
        
        COUNT(*) AS rating_count,
        COUNT(DISTINCT user_id) AS unique_user_count,
        ROUND(AVG(rating), 2) AS avg_rating,
        
        SUM(CASE WHEN rating >= 4.0 THEN 1 ELSE 0 END) AS positive_count,
        
        current_timestamp() AS dws_etl_time
    FROM dwd_fact_user_rating
    GROUP BY movie_id, DATE(event_time), year, month
""")

start = time.time()
dws_movie_daily.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet(str(DWS / 'dws_movie_daily'))
print(f'✅ dws_movie_daily 完成,耗时 {time.time()-start:.1f} 秒')

spark.read.parquet(str(DWS / 'dws_movie_daily')).createOrReplaceTempView('dws_movie_daily')

26/06/23 06:10:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:10:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:10:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:10:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:10:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:10:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:10:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:10:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:10:34 WARN RowBasedKeyValueBatch: Calling spill() on

✅ dws_movie_daily 完成,耗时 40.3 秒


## 1.3 dws_user_profile —— 用户画像表

**业务含义**：每个用户的累计画像,刻画"这个用户是怎样的人"。

**下游消费者**：
- `ads_user_funnel`（漏斗）会基于用户分层
- 未来 ML 推荐系统会用这张表做用户表征

**这张表的设计是面试高频题**——"你怎么设计一张用户画像表？"。

**关键决策**：
- 粒度是"每个用户一行"（不按时间分区）
- 包含**累计统计** + **行为偏好** + **分层标签**三类字段
- 用 PERCENTILE 算分位数,用 collect_list/sort_array 找用户最爱的类型

In [4]:
dws_user_profile = spark.sql("""
    WITH user_basic AS (
        -- 第一步:每个用户的基础统计
        SELECT 
            user_id,
            COUNT(*) AS total_ratings,
            COUNT(DISTINCT movie_id) AS unique_movies,
            COUNT(DISTINCT DATE(event_time)) AS active_days,
            MIN(event_time) AS first_rating_time,
            MAX(event_time) AS last_rating_time,
            DATEDIFF(MAX(event_time), MIN(event_time)) AS lifetime_days,
            ROUND(AVG(rating), 2) AS avg_rating_given,
            SUM(CASE WHEN sentiment = 'positive' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS positive_rate,
            SUM(CASE WHEN rating = 5.0 THEN 1 ELSE 0 END) AS five_star_count,
            SUM(CASE WHEN rating <= 1.0 THEN 1 ELSE 0 END) AS low_star_count
        FROM dwd_fact_user_rating
        GROUP BY user_id
    ),
    user_genre AS (
        -- 第二步:每个用户最常打分的类型 (通过 JOIN dim_movie 拿到 genres)
        SELECT 
            f.user_id,
            genre,
            COUNT(*) AS genre_rating_count,
            ROW_NUMBER() OVER (PARTITION BY f.user_id ORDER BY COUNT(*) DESC) AS rn
        FROM dwd_fact_user_rating f
        JOIN dwd_dim_movie m ON f.movie_id = m.movieId
        LATERAL VIEW EXPLODE(SPLIT(m.genres, ',')) g AS genre
        WHERE m.genres IS NOT NULL
        GROUP BY f.user_id, genre
    ),
    user_top_genre AS (
        -- 取每个用户最爱的 Top 1 类型
        SELECT user_id, genre AS favorite_genre
        FROM user_genre WHERE rn = 1
    )
    -- 主查询:拼接所有维度,并加上分层标签
    SELECT 
        b.user_id,
        b.total_ratings,
        b.unique_movies,
        b.active_days,
        b.first_rating_time,
        b.last_rating_time,
        b.lifetime_days,
        b.avg_rating_given,
        ROUND(b.positive_rate, 3) AS positive_rate,
        b.five_star_count,
        b.low_star_count,
        g.favorite_genre,
        
        -- 用户活跃度分层
        CASE 
            WHEN b.total_ratings >= 1000 THEN '1_super_active'
            WHEN b.total_ratings >= 100 THEN '2_active'
            WHEN b.total_ratings >= 20 THEN '3_normal'
            ELSE '4_low'
        END AS activity_tier,
        
        -- 评分严苛度分层
        CASE 
            WHEN b.low_star_count * 1.0 / b.total_ratings > 0.10 THEN 'severe'
            WHEN b.positive_rate > 0.80 THEN 'lenient'
            ELSE 'normal'
        END AS critic_type,
        
        current_timestamp() AS dws_etl_time
    FROM user_basic b
    LEFT JOIN user_top_genre g ON b.user_id = g.user_id
""")

start = time.time()
dws_user_profile.write.mode("overwrite").parquet(str(DWS / 'dws_user_profile'))
print(f'✅ dws_user_profile 完成,耗时 {time.time()-start:.1f} 秒')

spark.read.parquet(str(DWS / 'dws_user_profile')).createOrReplaceTempView('dws_user_profile')
spark.sql("SELECT * FROM dws_user_profile WHERE total_ratings >= 100 LIMIT 5").show(truncate=False)

✅ dws_user_profile 完成,耗时 31.9 秒
+-------+-------------+-------------+-----------+-------------------+-------------------+-------------+----------------+-------------+---------------+--------------+--------------+-------------+-----------+--------------------------+
|user_id|total_ratings|unique_movies|active_days|first_rating_time  |last_rating_time   |lifetime_days|avg_rating_given|positive_rate|five_star_count|low_star_count|favorite_genre|activity_tier|critic_type|dws_etl_time              |
+-------+-------------+-------------+-----------+-------------------+-------------------+-------------+----------------+-------------+---------------+--------------+--------------+-------------+-----------+--------------------------+
|3      |656          |656          |5          |2015-08-13 06:23:19|2019-08-17 18:31:23|1465         |3.7             |0.544        |32             |0             |Action        |2_active     |normal     |2026-06-23 06:11:04.388658|
|5      |101          |101      

---
# Part 2 · ADS 层建设

ADS 表的设计原则:**一张表对应前端的一个看板/图表**。命名规范用 `ads_业务名` 而不是 `ads_技术名`。

## 2.1 ads_dau_monthly —— 月度活跃趋势

**前端用途**：首页"平台月活趋势"折线图。

In [5]:
ads_dau_monthly = spark.sql("""
    SELECT 
        year,
        month,
        CONCAT(CAST(year AS STRING), '-', LPAD(CAST(month AS STRING), 2, '0')) AS year_month,
        COUNT(DISTINCT user_id) AS mau,
        SUM(rating_count) AS total_ratings,
        ROUND(AVG(avg_rating), 2) AS platform_avg_rating
    FROM dws_user_daily
    GROUP BY year, month
    ORDER BY year, month
""")

ads_dau_monthly.write.mode("overwrite").parquet(str(ADS / 'ads_dau_monthly'))
ads_dau_monthly.createOrReplaceTempView('ads_dau_monthly')
print(f'✅ ads_dau_monthly: {ads_dau_monthly.count()} 行')
ads_dau_monthly.show(5)

✅ ads_dau_monthly: 287 行
+----+-----+----------+----+-------------+-------------------+
|year|month|year_month| mau|total_ratings|platform_avg_rating|
+----+-----+----------+----+-------------+-------------------+
|1995|    1|   1995-01|   1|            3|               3.67|
|1996|    1|   1996-01|   6|           42|               3.77|
|1996|    2|   1996-02|  37|         1072|               3.84|
|1996|    3|   1996-03| 207|         7498|               3.94|
|1996|    4|   1996-04|1147|        46773|               3.82|
+----+-----+----------+----+-------------+-------------------+
only showing top 5 rows



## 2.2 ads_top_movies —— Top 内容榜

**前端用途**：首页"Top 100 内容"卡片墙。

**业务设计思考**：怎么定义"Top"？单看评分会让"5 个人打 5 分"这种榜上无名作品上来。所以用 **"贝叶斯加权评分"** 的简化版——评分 × 投票数权重。

In [6]:
ads_top_movies = spark.sql("""
    WITH movie_stats AS (
        SELECT 
            f.movie_id,
            COUNT(*) AS ml_rating_count,
            COUNT(DISTINCT f.user_id) AS unique_users,
            ROUND(AVG(f.rating), 2) AS ml_avg_rating,
            SUM(CASE WHEN f.rating >= 4.0 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS positive_rate
        FROM dwd_fact_user_rating f
        GROUP BY f.movie_id
        HAVING COUNT(*) >= 100  -- 至少 100 人评过
    )
    SELECT 
        s.movie_id,
        m.title,
        m.release_year,
        m.genres,
        m.imdb_id,
        m.tmdb_id,
        m.era,
        m.quality_tier,
        s.ml_rating_count,
        s.unique_users,
        s.ml_avg_rating,
        m.imdb_rating,
        m.imdb_votes,
        ROUND(s.positive_rate, 3) AS positive_rate,
        
        -- 综合排序得分:评分 × log(投票数),既看口碑又看热度
        ROUND(s.ml_avg_rating * LOG10(s.ml_rating_count + 1), 3) AS popularity_score,
        
        ROW_NUMBER() OVER (ORDER BY s.ml_avg_rating * LOG10(s.ml_rating_count + 1) DESC) AS overall_rank
    FROM movie_stats s
    JOIN dwd_dim_movie m ON s.movie_id = m.movieId
    WHERE m.title IS NOT NULL
    ORDER BY popularity_score DESC
    LIMIT 100
""")

ads_top_movies.write.mode("overwrite").parquet(str(ADS / 'ads_top_movies'))
ads_top_movies.createOrReplaceTempView('ads_top_movies')
print(f'✅ ads_top_movies: {ads_top_movies.count()} 行')
ads_top_movies.select('overall_rank', 'title', 'release_year', 'ml_avg_rating', 'unique_users', 'popularity_score').show(10, truncate=False)

26/06/23 06:11:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:11:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:11:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:11:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:11:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:11:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:11:57 WARN RowBasedKeyValueBatch: Calling spil

✅ ads_top_movies: 100 行


26/06/23 06:12:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:12:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:12:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:12:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:12:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:12:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:12:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:12:12 WARN RowBasedKeyVal

+------------+----------------------------------+------------+-------------+------------+----------------+
|overall_rank|title                             |release_year|ml_avg_rating|unique_users|popularity_score|
+------------+----------------------------------+------------+-------------+------------+----------------+
|1           |The Shawshank Redemption          |1994        |4.41         |81482       |21.658          |
|2           |Pulp Fiction                      |1994        |4.19         |79672       |20.536          |
|3           |The Godfather                     |1972        |4.32         |52498       |20.391          |
|4           |Schindler's List                  |1993        |4.25         |60411       |20.32           |
|5           |The Usual Suspects                |1995        |4.28         |55366       |20.301          |
|6           |The Silence of the Lambs          |1991        |4.15         |74127       |20.21           |
|7           |The Matrix             

## 2.3 ads_user_retention —— Cohort 留存矩阵 ⭐ 重点

**前端用途**：网站"用户留存分析"热力图。

**这是数据岗最高频面试题之一**:"留存率怎么算？"今天你建的这张表,**就是教科书答案的产物**。

**Cohort 留存的逻辑**：
1. 按用户首次活跃年份分组（cohort）
2. 看每个 cohort 在后续 N 年还有多少人活跃
3. 算留存比例

结果是一个矩阵：行是首年,列是 0/1/2/3/...年后,值是留存用户数和留存率。

In [7]:
ads_user_retention = spark.sql("""
    WITH user_first_year AS (
        -- 每个用户首次活跃年
        SELECT user_id, MIN(year) AS cohort_year
        FROM dwd_fact_user_rating
        GROUP BY user_id
    ),
    user_active_years AS (
        -- 每个用户活跃过的所有年份
        SELECT DISTINCT user_id, year AS active_year
        FROM dwd_fact_user_rating
    ),
    retention_matrix AS (
        SELECT 
            f.cohort_year,
            a.active_year - f.cohort_year AS year_offset,
            COUNT(DISTINCT a.user_id) AS active_users
        FROM user_first_year f
        JOIN user_active_years a ON f.user_id = a.user_id
        WHERE f.cohort_year >= 2000 
          AND a.active_year >= f.cohort_year
          AND a.active_year - f.cohort_year <= 5  -- 看前 5 年的留存
        GROUP BY f.cohort_year, a.active_year - f.cohort_year
    ),
    cohort_size AS (
        -- 每个 cohort 的初始用户数(即 year_offset=0 的人数)
        SELECT cohort_year, active_users AS cohort_total
        FROM retention_matrix
        WHERE year_offset = 0
    )
    SELECT 
        r.cohort_year,
        r.year_offset,
        r.active_users,
        c.cohort_total,
        ROUND(100.0 * r.active_users / c.cohort_total, 1) AS retention_pct
    FROM retention_matrix r
    JOIN cohort_size c ON r.cohort_year = c.cohort_year
    ORDER BY r.cohort_year, r.year_offset
""")

ads_user_retention.write.mode("overwrite").parquet(str(ADS / 'ads_user_retention'))
ads_user_retention.createOrReplaceTempView('ads_user_retention')

print(f'✅ ads_user_retention: {ads_user_retention.count()} 行')
print('\n📊 留存矩阵预览(行=首年, 列=年偏移, 值=留存率%):')
spark.sql("""
    SELECT 
        cohort_year,
        MAX(CASE WHEN year_offset = 0 THEN retention_pct END) AS y0,
        MAX(CASE WHEN year_offset = 1 THEN retention_pct END) AS y1,
        MAX(CASE WHEN year_offset = 2 THEN retention_pct END) AS y2,
        MAX(CASE WHEN year_offset = 3 THEN retention_pct END) AS y3,
        MAX(CASE WHEN year_offset = 4 THEN retention_pct END) AS y4,
        MAX(CASE WHEN year_offset = 5 THEN retention_pct END) AS y5
    FROM ads_user_retention
    GROUP BY cohort_year
    ORDER BY cohort_year
""").show()

✅ ads_user_retention: 105 行

📊 留存矩阵预览(行=首年, 列=年偏移, 值=留存率%):
+-----------+-----+----+----+----+----+----+
|cohort_year|   y0|  y1|  y2|  y3|  y4|  y5|
+-----------+-----+----+----+----+----+----+
|       2000|100.0|16.3| 9.4| 7.1| 5.7| 4.7|
|       2001|100.0|16.1| 9.9| 7.3| 5.4| 4.1|
|       2002|100.0|19.4|12.0| 8.2| 6.3| 4.6|
|       2003|100.0|21.7|12.1| 8.7| 6.6| 5.5|
|       2004|100.0|20.8|12.4| 9.1| 6.8| 5.4|
|       2005|100.0|17.7|10.8| 7.5| 5.4| 4.7|
|       2006|100.0|19.2|11.3| 8.4| 6.8| 5.2|
|       2007|100.0|21.9|12.6|10.0| 7.4| 5.8|
|       2008|100.0|20.2|12.5| 9.1| 6.8| 5.2|
|       2009|100.0|18.5|11.0| 8.3| 6.0| 4.6|
|       2010|100.0|20.3|11.5| 8.3| 5.9| 4.6|
|       2011|100.0|16.6| 9.9| 6.9| 5.8| 4.7|
|       2012|100.0|15.0| 8.1| 6.5| 5.7| 4.5|
|       2013|100.0|13.7| 9.5| 7.1| 5.8| 4.9|
|       2014|100.0|17.1|10.7| 7.8| 6.0| 4.5|
|       2015|100.0|19.4|10.9| 9.6| 5.9|NULL|
|       2016|100.0|16.4|11.5| 6.9|NULL|NULL|
|       2017|100.0|19.6|10.5|NULL|NULL|N

**这张表的资历值**：你可以在简历项目里写：

> 实现 Cohort 留存分析模型 (`ads_user_retention`),按用户首次活跃年份分组,产出 6 年留存率矩阵。

**面试时会问**："如果让你算"首日留存""7 日留存",这套逻辑怎么改？"答案：把 cohort_year 换成 cohort_date,把 year_offset 换成 date_offset,逻辑完全一样。

## 2.4 ads_genre_popularity —— 类型受欢迎度

**前端用途**："类型偏好"热力图（年代 × 类型）。

In [8]:
ads_genre_popularity = spark.sql("""
    SELECT 
        m.era,
        genre,
        COUNT(*) AS rating_count,
        COUNT(DISTINCT f.user_id) AS unique_users,
        COUNT(DISTINCT f.movie_id) AS unique_movies,
        ROUND(AVG(f.rating), 2) AS avg_rating,
        SUM(CASE WHEN f.rating >= 4.0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS positive_pct
    FROM dwd_fact_user_rating f
    JOIN dwd_dim_movie m ON f.movie_id = m.movieId
    LATERAL VIEW EXPLODE(SPLIT(m.genres, ',')) g AS genre
    WHERE m.genres IS NOT NULL
      AND m.era != '未知'
    GROUP BY m.era, genre
""")

ads_genre_popularity.write.mode("overwrite").parquet(str(ADS / 'ads_genre_popularity'))
ads_genre_popularity.createOrReplaceTempView('ads_genre_popularity')
print(f'✅ ads_genre_popularity: {ads_genre_popularity.count()} 行')

✅ ads_genre_popularity: 124 行


## 2.5 ads_user_funnel —— 参与度漏斗 ⭐ 跟你易多保经历呼应

**前端用途**："用户参与度漏斗"。

**业务意义**：这张表回答"我们的用户参与度像什么形状"——从轻度用户到重度用户的转化情况。**直接对应你易多保实习里设计的"浏览→加购→下单"漏斗思想**,放到这里就是"打 1 分→打 10 分→打 50 分→打 100 分"。

In [9]:
ads_user_funnel = spark.sql("""
    WITH base AS (
        SELECT COUNT(*) AS total_users FROM dws_user_profile
    ),
    funnel AS (
        SELECT '1_注册用户' AS stage, COUNT(*) AS user_count, 1 AS step FROM dws_user_profile
        UNION ALL
        SELECT '2_打过 ≥10 部', COUNT(*), 2 FROM dws_user_profile WHERE total_ratings >= 10
        UNION ALL
        SELECT '3_打过 ≥50 部', COUNT(*), 3 FROM dws_user_profile WHERE total_ratings >= 50
        UNION ALL
        SELECT '4_打过 ≥100 部', COUNT(*), 4 FROM dws_user_profile WHERE total_ratings >= 100
        UNION ALL
        SELECT '5_打过 ≥500 部', COUNT(*), 5 FROM dws_user_profile WHERE total_ratings >= 500
        UNION ALL
        SELECT '6_打过 ≥1000 部', COUNT(*), 6 FROM dws_user_profile WHERE total_ratings >= 1000
    )
    SELECT 
        f.stage,
        f.step,
        f.user_count,
        ROUND(100.0 * f.user_count / b.total_users, 2) AS overall_pct,
        ROUND(100.0 * f.user_count / LAG(f.user_count) OVER (ORDER BY f.step), 2) AS step_conv_rate
    FROM funnel f
    CROSS JOIN base b
    ORDER BY f.step
""")

ads_user_funnel.write.mode("overwrite").parquet(str(ADS / 'ads_user_funnel'))
ads_user_funnel.createOrReplaceTempView('ads_user_funnel')
print(f'✅ ads_user_funnel:')
ads_user_funnel.show()

26/06/23 06:13:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 0

✅ ads_user_funnel:


26/06/23 06:13:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 0

+---------------+----+----------+-----------+--------------+
|          stage|step|user_count|overall_pct|step_conv_rate|
+---------------+----+----------+-----------+--------------+
|     1_注册用户|   1|    162541|     100.00|          NULL|
|  2_打过 ≥10 部|   2|    162541|     100.00|        100.00|
|  3_打过 ≥50 部|   3|    102492|      63.06|         63.06|
| 4_打过 ≥100 部|   4|     63892|      39.31|         62.34|
| 5_打过 ≥500 部|   5|      9713|       5.98|         15.20|
|6_打过 ≥1000 部|   6|      2675|       1.65|         27.54|
+---------------+----+----------+-----------+--------------+



26/06/23 06:13:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


**看这个漏斗你能讲出什么故事**：
- 注册了的用户里,大约多少比例打过 10 部以上？
- 每一级的转化率（step_conv_rate）是多少？
- 从 100→500 这一级流失最严重吗？为什么？

**这就是数据分析师每天在做的事**——指标算出来不算完,**读出业务故事才算完**。

---
# Part 3 · 导出 JSON 给前端

ADS 表存成 Parquet 是数据工程内部用的,前端 JavaScript 读不了 Parquet。**这一步把 ADS 表导出成 JSON,Day 6 前端可以直接 fetch**。

**这是连接"数据工程"和"前端可见性"的关键一步**。

In [10]:
import json

def export_to_json(table_view_name, json_filename):
    """把一张 ADS 表导出成单个 JSON 文件,给前端用"""
    df = spark.sql(f"SELECT * FROM {table_view_name}")
    rows = [row.asDict() for row in df.collect()]
    
    # 处理日期/时间类型,转成字符串
    for row in rows:
        for k, v in row.items():
            if hasattr(v, 'isoformat'):
                row[k] = v.isoformat()
    
    output_path = ADS_JSON / json_filename
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False, indent=2, default=str)
    
    file_size_kb = output_path.stat().st_size / 1024
    print(f'  ✅ {json_filename:30s} {len(rows):6d} 行 / {file_size_kb:7.1f} KB')

print('📤 导出 ADS 表到 JSON:')
export_to_json('ads_dau_monthly', 'dau_monthly.json')
export_to_json('ads_top_movies', 'top_movies.json')
export_to_json('ads_user_retention', 'user_retention.json')
export_to_json('ads_genre_popularity', 'genre_popularity.json')
export_to_json('ads_user_funnel', 'user_funnel.json')

print(f'\n📂 输出目录: {ADS_JSON}')

📤 导出 ADS 表到 JSON:
  ✅ dau_monthly.json                  287 行 /    41.8 KB


26/06/23 06:13:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/23 06:13:19 WARN RowBasedKeyValueBatch: Calling spil

  ✅ top_movies.json                   100 行 /    44.0 KB
  ✅ user_retention.json               105 行 /    13.9 KB
  ✅ genre_popularity.json             124 行 /    24.5 KB


26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


  ✅ user_funnel.json                    6 行 /     0.8 KB

📂 输出目录: /home/rein/projects/content-data-platform/frontend/data


26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 06:13:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/23 0

**Day 6 你写前端时,直接 `fetch('./data/top_movies.json')` 就能用**。

JSON 而不是 CSV/Parquet 的原因：浏览器原生支持,跟 JS 数据结构无缝对接,ECharts 直接吃。

---
# Part 4 · 总览

看看你今天建好的完整数据仓库。

In [11]:
def get_dir_size(path):
    total = 0
    for p in Path(path).rglob('*'):
        if p.is_file():
            total += p.stat().st_size
    return total / 1024 / 1024

print('=' * 65)
print('🏛️  你的完整数据仓库')
print('=' * 65)

for layer_name, layer_path in [('ODS', WAREHOUSE / 'ods'), ('DWD', WAREHOUSE / 'dwd'),
                                ('DWS', DWS), ('ADS', ADS)]:
    if not layer_path.exists():
        continue
    tables = sorted([p for p in layer_path.iterdir() if p.is_dir()])
    layer_size = get_dir_size(layer_path)
    print(f'\n📂 {layer_name} 层  ({layer_size:.1f} MB)')
    for t in tables:
        partition_count = len([p for p in t.iterdir() if p.is_dir() and '=' in p.name])
        marker = f'(✨ {partition_count} 分区)' if partition_count > 0 else ''
        print(f'    {t.name:30s} {get_dir_size(t):7.1f} MB  {marker}')

print(f'\n📂 前端数据 JSON  ({get_dir_size(ADS_JSON):.2f} MB)')
for f in sorted(ADS_JSON.iterdir()):
    if f.suffix == '.json':
        print(f'    {f.name:30s} {f.stat().st_size / 1024:7.1f} KB')

print('\n' + '=' * 65)
print(f'总数据量: {get_dir_size(WAREHOUSE):.1f} MB')
print('=' * 65)

🏛️  你的完整数据仓库

📂 ODS 层  (602.5 MB)
    imdb_ratings                      12.4 MB  
    imdb_titles                      433.3 MB  
    ml_links                           0.7 MB  
    ml_movies                          1.5 MB  
    ml_ratings                       154.5 MB  

📂 DWD 层  (248.8 MB)
    dim_movie                          2.9 MB  
    fact_user_rating                 245.9 MB  (✨ 25 分区)

📂 DWS 层  (88.0 MB)
    dws_movie_daily                   67.2 MB  (✨ 25 分区)
    dws_user_daily                    14.7 MB  (✨ 25 分区)
    dws_user_profile                   6.1 MB  

📂 ADS 层  (0.0 MB)
    ads_dau_monthly                    0.0 MB  
    ads_genre_popularity               0.0 MB  
    ads_top_movies                     0.0 MB  
    ads_user_funnel                    0.0 MB  
    ads_user_retention                 0.0 MB  

📂 前端数据 JSON  (0.12 MB)
    dau_monthly.json                  41.8 KB
    genre_popularity.json             24.5 KB
    top_movies.json                   44.0 

In [12]:
spark.stop()
print('Spark 已关闭')

Spark 已关闭


---
# 🎉 Day 5 完成

## 你今天获得的能力

1. **DWS / ADS 设计思维** —— 你理解了"半成品仓库"和"出餐口"的区别,知道一个查询该放在哪一层
2. **用户画像建模** —— 你建了一张完整的 `dws_user_profile`,这是面试经常问的设计题
3. **Cohort 留存** —— 你不再是"听说过留存"的人,你是"亲手写过留存矩阵"的人
4. **漏斗分析** —— 易多保实习里设计的漏斗思想,今天在数据工程层面真正落地
5. **数据工程到前端的桥** —— 你输出了 5 个 JSON,Day 6 前端可以直接消费

## 简历可以补的新话术

> 基于离线数仓 DWD 层建设 DWS 用户画像表与日聚合表,产出 5 张 ADS 业务指标表,涵盖 DAU 趋势、Top 内容榜、Cohort 留存矩阵、类型受欢迎度、参与度漏斗,并通过 JSON 导出供前端可视化层消费。

## 明天 Day 6

**前端可见性日** —— 用你今天导出的 JSON,做出第一版可视化看板。Apple/喜茶风格的蓝绿色 + ECharts。明天结束你可以打开浏览器看到自己的网站雏形。